## NAICS 2 - Deberta Large - Run 3

In [1]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [2]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-large.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-large")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-large
Contents: ['snapshots', 'blobs', 'refs']


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import os
import gc
import re
import random
import shutil
import copy
from collections import OrderedDict

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [4]:
df_raw = pd.read_csv('ExioNAICS.csv')

SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
}

naics2_raw = df_raw[['NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description']].drop_duplicates(subset='NAICS_2 Code').copy()
naics2_raw['NAICS_2 Code'] = naics2_raw['NAICS_2 Code'].astype(str)
naics2_raw['sector_code'] = naics2_raw['NAICS_2 Code'].map(SECTOR_MERGE).fillna(naics2_raw['NAICS_2 Code'])

naics2_corpus = naics2_raw.drop_duplicates(subset='sector_code').copy()
naics2_corpus = naics2_corpus.sort_values('sector_code').reset_index(drop=True)

sector_codes  = naics2_corpus['sector_code'].tolist()
sector_titles = naics2_corpus['NAICS_2 Title'].tolist()
code_to_idx   = {code: i for i, code in enumerate(sector_codes)}
NUM_CLASSES   = len(sector_codes)
print(f"NAICS-2 sectors: {NUM_CLASSES}")


def clean_text_keep_case(text):
    if pd.isna(text):
        return ""
    s = str(text)
    s = re.sub(r'http\S+|www\.\S+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


real = df_raw[['Company Name', 'Company Description', 'NAICS Code']].copy()
real['NAICS Code'] = real['NAICS Code'].astype(str)
real['naics2_raw'] = real['NAICS Code'].str[:2]
real['sector'] = real['naics2_raw'].map(SECTOR_MERGE).fillna(real['naics2_raw'])
real['naics2_idx'] = real['sector'].map(code_to_idx)
real = real.dropna(subset=['naics2_idx', 'Company Description']).reset_index(drop=True)
real['naics2_idx'] = real['naics2_idx'].astype(int)

real['name_clean'] = real['Company Name'].apply(clean_text_keep_case)
real['desc_clean'] = real['Company Description'].apply(clean_text_keep_case)
real['query'] = (real['name_clean'] + '. ' + real['desc_clean']).str.strip('. ').str.strip()
real = real[real['query'].str.len() >= 10].reset_index(drop=True)
real = real.drop_duplicates(subset=['query']).reset_index(drop=True)

print(f"Real samples (cased): {len(real)}")
print(f"Sample query: {real['query'].iloc[0][:200]}")


print("\n=== Building NAICS-6/5/4/3 augmentation training data (cased) ===")
naics_sources = [
    ('NAICS-6', 'NAICS Code',   'NAICS Title',   'Description'),
    ('NAICS-5', 'NAICS_5 Code', 'NAICS_5 Title', 'NAICS_5 Description'),
    ('NAICS-4', 'NAICS_4 Code', 'NAICS_4 Title', 'NAICS_4 Description'),
    ('NAICS-3', 'NAICS_3 Code', 'NAICS_3 Title', 'NAICS_3 Description'),
]

aug_rows = []
for level, code_col, title_col, desc_col in naics_sources:
    sub = df_raw[[code_col, title_col, desc_col]].drop_duplicates(subset=code_col).dropna(subset=[code_col])
    for _, row in sub.iterrows():
        code  = str(row[code_col])
        title = "" if pd.isna(row[title_col]) else str(row[title_col])
        desc  = "" if pd.isna(row[desc_col])  else str(row[desc_col])
        text = clean_text_keep_case((title + ". " + desc).strip())
        if len(text) < 20:
            continue
        naics2_prefix = code[:2]
        sector = SECTOR_MERGE.get(naics2_prefix, naics2_prefix)
        if sector not in code_to_idx:
            continue
        aug_rows.append({
            'naics_code': code,
            'level': level,
            'query': text,
            'sector': sector,
            'naics2_idx': code_to_idx[sector],
        })

df_aug = pd.DataFrame(aug_rows).drop_duplicates(subset='naics_code').reset_index(drop=True)
print(f"Aug samples: {len(df_aug)}  (will be down-weighted to 0.5 in loss)")


print("\n=== Enriching NAICS-2 corpus with NAICS-6 example titles (cased) ===")
MAX_EXAMPLES_PER_SECTOR = 50

naics6_titles = df_raw[['NAICS Code', 'NAICS Title']].drop_duplicates(subset='NAICS Code').dropna()
naics6_titles['NAICS Code'] = naics6_titles['NAICS Code'].astype(str)
naics6_titles['sector'] = naics6_titles['NAICS Code'].str[:2].map(SECTOR_MERGE).fillna(naics6_titles['NAICS Code'].str[:2])

sector_to_examples = {}
for sector, grp in naics6_titles.groupby('sector'):
    titles = [t.strip() for t in grp['NAICS Title'].astype(str).tolist() if t.strip()]
    sector_to_examples[sector] = titles[:MAX_EXAMPLES_PER_SECTOR]

corpus_texts = []
for i, code in enumerate(sector_codes):
    title = sector_titles[i]
    description = naics2_corpus['NAICS_2 Description'].iloc[i]
    description = "" if not pd.notna(description) else str(description)
    description = clean_text_keep_case(description)
    examples = sector_to_examples.get(code, [])
    if examples:
        text = f"{title}. Examples: {'; '.join(examples)}. {description}"
    else:
        text = f"{title}. {description}"
    corpus_texts.append(text)

print(f"Enriched corpus length (chars): "
      f"min={min(len(t) for t in corpus_texts)}, "
      f"max={max(len(t) for t in corpus_texts)}, "
      f"mean={int(np.mean([len(t) for t in corpus_texts]))}")

del df_raw
gc.collect()

NAICS-2 sectors: 20
Real samples (cased): 14175
Sample query: Foot Locker Specialty Inc. Foot Locker Inc (Foot Locker) is a specialty retailer of athletic footwear and apparel

=== Building NAICS-6/5/4/3 augmentation training data (cased) ===
Aug samples: 2173  (will be down-weighted to 0.5 in loss)

=== Enriching NAICS-2 corpus with NAICS-6 example titles (cased) ===
Enriched corpus length (chars): min=1133, max=9157, mean=3750


60

In [5]:
MODEL_NAME       = "microsoft/deberta-v3-large"
MAX_LENGTH       = 224
BATCH_SIZE       = 4
GRAD_ACCUM       = 2
NUM_EPOCHS       = 8
LEARNING_RATE    = 1.5e-5
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01
VAL_RATIO        = 0.10
PATIENCE         = 3

LABEL_SMOOTHING  = 0.10
HEAD_DROPOUT     = 0.10
HIDDEN_DROPOUT   = 0.10
ATTN_DROPOUT     = 0.10
NUM_MSD_SAMPLES  = 1

LLRD_DECAY       = 0.90

USE_FGM          = True
FGM_EPSILON      = 0.5

USE_HARD_NEG     = True
MARGIN           = 0.5
LAMBDA_MARGIN    = 0.20

USE_EMA          = True
EMA_DECAY        = 0.999

USE_SWA          = True
SWA_START_EPOCH  = 4

AUG_WEIGHT       = 0.5
MAX_GRAD_NORM    = 1.0

SEED             = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"=== NAICS-2 Cross-Encoder LAST RUN (seed={SEED}) ===")
print(f"  Model:           {MODEL_NAME}")
print(f"  Max length:      {MAX_LENGTH}")
print(f"  Batch / accum:   {BATCH_SIZE} x {GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"                   ({BATCH_SIZE*NUM_CLASSES} sequences per fwd)")
print(f"  Epochs:          {NUM_EPOCHS}  (early stop patience {PATIENCE})")
print(f"  LR:              {LEARNING_RATE}  (LLRD decay {LLRD_DECAY}, warmup {WARMUP_RATIO})")
print(f"  Dropouts:        head={HEAD_DROPOUT} hidden={HIDDEN_DROPOUT} attn={ATTN_DROPOUT}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  FGM:             {USE_FGM} (eps={FGM_EPSILON})")
print(f"  Hard-neg margin: {USE_HARD_NEG} (m={MARGIN}, lambda={LAMBDA_MARGIN})")
print(f"  EMA weights:     {USE_EMA} (decay={EMA_DECAY})")
print(f"  SWA averaging:   {USE_SWA} (start epoch {SWA_START_EPOCH})")
print(f"  Aug weight:      {AUG_WEIGHT}")

=== NAICS-2 Cross-Encoder LAST RUN (seed=42) ===
  Model:           microsoft/deberta-v3-large
  Max length:      224
  Batch / accum:   4 x 2 = effective 8
                   (80 sequences per fwd)
  Epochs:          8  (early stop patience 3)
  LR:              1.5e-05  (LLRD decay 0.9, warmup 0.06)
  Dropouts:        head=0.1 hidden=0.1 attn=0.1
  Label smoothing: 0.1
  FGM:             True (eps=0.5)
  Hard-neg margin: True (m=0.5, lambda=0.2)
  EMA weights:     True (decay=0.999)
  SWA averaging:   True (start epoch 4)
  Aug weight:      0.5


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=1)
config.hidden_dropout_prob          = HIDDEN_DROPOUT
config.attention_probs_dropout_prob = ATTN_DROPOUT
if hasattr(config, 'cls_dropout'):
    config.cls_dropout = HEAD_DROPOUT

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config, torch_dtype=torch.float32
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_params:,}")
print(f"Encoder layers: {config.num_hidden_layers}, hidden: {config.hidden_size}")
print(f"Hidden dropout: {config.hidden_dropout_prob}, Attn dropout: {config.attention_probs_dropout_prob}")

The tokenizer you are loading from 'microsoft/deberta-v3-large' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Total params: 435,062,785
Encoder layers: 24, hidden: 1024
Hidden dropout: 0.1, Attn dropout: 0.1


In [7]:
def pretokenize(queries, num_classes, max_length, label_name):
    ids, masks = [], []
    for i, q in enumerate(queries):
        enc = tokenizer(
            [q] * num_classes,
            corpus_texts,
            max_length=max_length, truncation=True,
            padding='max_length', return_tensors='pt',
        )
        ids.append(enc['input_ids'].to(torch.int32))
        masks.append(enc['attention_mask'].to(torch.int8))
        if (i + 1) % 2000 == 0:
            print(f"  [{label_name}] {i+1}/{len(queries)} done")
    return torch.stack(ids), torch.stack(masks)


print(f"Pre-tokenizing {len(real)} real samples x {NUM_CLASSES} candidates...")
all_input_ids, all_attention_masks = pretokenize(
    real['query'].tolist(), NUM_CLASSES, MAX_LENGTH, "real"
)
all_labels = torch.tensor(real['naics2_idx'].values, dtype=torch.long)
print(f"Real pre-tokenized shape: {all_input_ids.shape}  dtype={all_input_ids.dtype}")

print(f"\nPre-tokenizing {len(df_aug)} augmentation samples x {NUM_CLASSES} candidates...")
aug_input_ids, aug_attention_masks = pretokenize(
    df_aug['query'].tolist(), NUM_CLASSES, MAX_LENGTH, "aug"
)
aug_labels = torch.tensor(df_aug['naics2_idx'].values, dtype=torch.long)
print(f"Aug pre-tokenized shape: {aug_input_ids.shape}")

total_mem = (all_input_ids.nbytes + all_attention_masks.nbytes +
             aug_input_ids.nbytes + aug_attention_masks.nbytes) / 1e9
print(f"\nTotal memory: {total_mem:.2f} GB")


class PreTokenizedDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels, weights):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels
        self.weights = weights
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx].long(),
            'attention_mask': self.attention_masks[idx].long(),
            'label':          self.labels[idx],
            'weight':         self.weights[idx],
        }

print("Pre-tokenization complete.")

Pre-tokenizing 14175 real samples x 20 candidates...
  [real] 2000/14175 done
  [real] 4000/14175 done
  [real] 6000/14175 done
  [real] 8000/14175 done
  [real] 10000/14175 done
  [real] 12000/14175 done
  [real] 14000/14175 done
Real pre-tokenized shape: torch.Size([14175, 20, 224])  dtype=torch.int32

Pre-tokenizing 2173 augmentation samples x 20 candidates...
  [aug] 2000/2173 done
Aug pre-tokenized shape: torch.Size([2173, 20, 224])

Total memory: 0.37 GB
Pre-tokenization complete.


In [8]:
train_idx, val_idx = train_test_split(
    np.arange(len(real)), test_size=VAL_RATIO, random_state=SEED,
    stratify=real['naics2_idx'].values,
)

real_train_weights = torch.ones(len(train_idx), dtype=torch.float)
aug_weights        = torch.full((len(df_aug),), AUG_WEIGHT, dtype=torch.float)

train_input_ids       = torch.cat([all_input_ids[train_idx], aug_input_ids], dim=0)
train_attention_masks = torch.cat([all_attention_masks[train_idx], aug_attention_masks], dim=0)
train_labels          = torch.cat([all_labels[train_idx], aug_labels], dim=0)
train_weights         = torch.cat([real_train_weights, aug_weights], dim=0)

train_dataset = PreTokenizedDataset(train_input_ids, train_attention_masks,
                                    train_labels, train_weights)

val_queries = real['query'].iloc[val_idx].tolist()
val_labels_list = real['naics2_idx'].iloc[val_idx].tolist()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)

print(f"Real training samples:  {len(train_idx)}  (weight=1.0)")
print(f"Aug training samples:   {len(df_aug)}  (weight={AUG_WEIGHT})")
print(f"Total training samples: {len(train_dataset)}  ({len(train_loader)} batches)")
print(f"Val samples:            {len(val_queries)}  (real only)")
print(f"Effective batch size:   {BATCH_SIZE * GRAD_ACCUM}")
print(f"Optimizer steps/epoch:  {(len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM}")

Real training samples:  12757  (weight=1.0)
Aug training samples:   2173  (weight=0.5)
Total training samples: 14930  (3733 batches)
Val samples:            1418  (real only)
Effective batch size:   8
Optimizer steps/epoch:  1867


In [9]:
class FGM:
    def __init__(self, model, epsilon=1.0, emb_name='word_embeddings'):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                if name in self.backup:
                    param.data = self.backup[name]
        self.backup = {}


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self, model):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(d).add_(p.data, alpha=1 - d)

    def apply_shadow(self, model):
        self.backup = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                p.data.copy_(self.shadow[n])

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data.copy_(self.backup[n])
        self.backup = {}


class SWAState:
    def __init__(self):
        self.shadow = None
        self.n = 0
        self.backup = None

    def update(self, model):
        sd = model.state_dict()
        if self.shadow is None:
            self.shadow = {k: v.detach().cpu().clone().to(torch.float32) for k, v in sd.items()}
            self.n = 1
            return
        self.n += 1
        for k, v in sd.items():
            self.shadow[k].mul_(1 - 1.0 / self.n).add_(v.detach().cpu().to(torch.float32), alpha=1.0 / self.n)

    def has_data(self):
        return self.shadow is not None and self.n > 0

    def apply(self, model):
        sd = model.state_dict()
        self.backup = {k: v.detach().cpu().clone() for k, v in sd.items()}
        for k, v in sd.items():
            if k in self.shadow:
                target_dtype = v.dtype
                v.copy_(self.shadow[k].to(target_dtype).to(v.device))

    def restore(self, model):
        if self.backup is None:
            return
        sd = model.state_dict()
        for k, v in sd.items():
            if k in self.backup:
                v.copy_(self.backup[k].to(v.device))
        self.backup = None


def weighted_listwise_loss(scores, labels, sample_weights, class_weights,
                           label_smoothing=0.10, use_hard_neg=True,
                           margin=0.5, lambda_margin=0.20):
    sw = sample_weights / sample_weights.sum().clamp(min=1e-6) * sample_weights.numel()

    ce_per_sample = F.cross_entropy(
        scores, labels,
        weight=class_weights,
        label_smoothing=label_smoothing,
        reduction='none',
    )
    ce_loss = (ce_per_sample * sw).mean()

    if not use_hard_neg:
        return ce_loss, ce_loss, torch.tensor(0.0, device=scores.device)

    pos_scores = scores.gather(1, labels.unsqueeze(1)).squeeze(1)
    neg_scores = scores.clone()
    neg_scores.scatter_(1, labels.unsqueeze(1), float('-inf'))
    hardest_neg = neg_scores.max(dim=1).values
    margin_per_sample = F.relu(margin - pos_scores + hardest_neg)
    margin_loss = (margin_per_sample * sw).mean()

    total = ce_loss + lambda_margin * margin_loss
    return total, ce_loss, margin_loss


def get_llrd_param_groups(model, base_lr, weight_decay, decay_rate=0.90):
    no_decay = ['bias', 'LayerNorm.weight']
    num_layers = model.config.num_hidden_layers

    param_groups = []
    seen = set()

    for n, p in model.named_parameters():
        if 'embeddings' in n and 'rel_embeddings' not in n and id(p) not in seen:
            wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
            param_groups.append({
                'params': [p],
                'lr': base_lr * (decay_rate ** (num_layers + 1)),
                'weight_decay': wd,
            })
            seen.add(id(p))

    for layer_id in range(num_layers):
        prefix = f'encoder.layer.{layer_id}.'
        for n, p in model.named_parameters():
            if prefix in n and id(p) not in seen:
                wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
                param_groups.append({
                    'params': [p],
                    'lr': base_lr * (decay_rate ** (num_layers - layer_id - 1)),
                    'weight_decay': wd,
                })
                seen.add(id(p))

    for n, p in model.named_parameters():
        if id(p) not in seen:
            wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
            param_groups.append({
                'params': [p],
                'lr': base_lr,
                'weight_decay': wd,
            })
            seen.add(id(p))

    print(f"LLRD: {len(param_groups)} param groups | "
          f"min lr {min(g['lr'] for g in param_groups):.2e}, "
          f"max lr {max(g['lr'] for g in param_groups):.2e}")
    return param_groups


print("Defined: FGM, ModelEMA, SWAState, weighted_listwise_loss, get_llrd_param_groups")

Defined: FGM, ModelEMA, SWAState, weighted_listwise_loss, get_llrd_param_groups


In [10]:
@torch.no_grad()
def evaluate(model, tokenizer, queries, labels, corpus_texts, max_length, score_batch=64):
    model.eval()
    n = len(queries)
    num_classes = len(corpus_texts)
    top1, top3, top5 = 0, 0, 0
    all_preds = []

    for i in range(n):
        query = queries[i]
        true_label = labels[i]
        scores = []

        for j in range(0, num_classes, score_batch):
            batch_docs = corpus_texts[j:j+score_batch]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt',
            )
            ids   = enc['input_ids'].to(device)
            masks = enc['attention_mask'].to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(input_ids=ids, attention_mask=masks).logits.squeeze(-1)
            scores.extend(logits.float().cpu().tolist())

        scores_t = torch.tensor(scores)
        topk = scores_t.topk(min(5, num_classes)).indices.tolist()
        all_preds.append(topk[0])

        if topk[0] == true_label: top1 += 1
        if true_label in topk[:3]: top3 += 1
        if true_label in topk[:5]: top5 += 1

    macro_f1    = f1_score(labels, all_preds, average='macro',    zero_division=0)
    weighted_f1 = f1_score(labels, all_preds, average='weighted', zero_division=0)
    return {
        'top1': top1 / n, 'top3': top3 / n, 'top5': top5 / n,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'all_preds': all_preds,
    }


print("Evaluation function defined.")

Evaluation function defined.


In [11]:
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import time

CHECKPOINT_DIR = f"/content/drive/MyDrive/naics2_large_run3_checkpoints_seed{SEED}"
RESULTS_DIR    = f"results_naics2_large_run3_seed{SEED}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

eff_counts = np.zeros(NUM_CLASSES, dtype=np.float64)
labels_np  = train_labels.numpy()
weights_np = train_weights.numpy().astype(np.float64)
for c in range(NUM_CLASSES):
    eff_counts[c] = weights_np[labels_np == c].sum()

class_weights_np = 1.0 / np.sqrt(np.maximum(eff_counts, 1.0))
class_weights_np = class_weights_np / class_weights_np.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights_np, dtype=torch.float, device=device)

label_counts = np.bincount(labels_np, minlength=NUM_CLASSES)
print("Class weights (sqrt inv-effective-freq, normalized to mean=1):")
for code, title, cnt, ec, w in zip(sector_codes, sector_titles, label_counts, eff_counts, class_weights_np):
    print(f"  {code:>5}  raw={int(cnt):>5}  eff={ec:>7.1f}  weight={w:.3f}  {title[:46]}")

param_groups = get_llrd_param_groups(model, LEARNING_RATE, WEIGHT_DECAY, LLRD_DECAY)
optimizer = torch.optim.AdamW(param_groups)

opt_steps_per_epoch = (len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM
total_steps  = opt_steps_per_epoch * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

fgm = FGM(model, epsilon=FGM_EPSILON, emb_name='word_embeddings') if USE_FGM else None
ema = ModelEMA(model, decay=EMA_DECAY) if USE_EMA else None
swa = SWAState() if USE_SWA else None

print(f"\nOpt steps/epoch: {opt_steps_per_epoch}  | total opt steps: {total_steps}  | warmup: {warmup_steps}")

best_top1 = 0.0
best_epoch = 0
best_strategy = "none"
patience_counter = 0
epoch_log = []

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")

config_dict = {
    'MODEL_NAME': MODEL_NAME,
    'MAX_LENGTH': MAX_LENGTH, 'BATCH_SIZE': BATCH_SIZE, 'GRAD_ACCUM': GRAD_ACCUM,
    'NUM_EPOCHS': NUM_EPOCHS, 'LEARNING_RATE': LEARNING_RATE,
    'WARMUP_RATIO': WARMUP_RATIO, 'WEIGHT_DECAY': WEIGHT_DECAY,
    'LABEL_SMOOTHING': LABEL_SMOOTHING,
    'HEAD_DROPOUT': HEAD_DROPOUT, 'HIDDEN_DROPOUT': HIDDEN_DROPOUT, 'ATTN_DROPOUT': ATTN_DROPOUT,
    'NUM_MSD_SAMPLES': NUM_MSD_SAMPLES,
    'LLRD_DECAY': LLRD_DECAY,
    'USE_FGM': USE_FGM, 'FGM_EPSILON': FGM_EPSILON,
    'USE_HARD_NEG': USE_HARD_NEG, 'MARGIN': MARGIN, 'LAMBDA_MARGIN': LAMBDA_MARGIN,
    'USE_EMA': USE_EMA, 'EMA_DECAY': EMA_DECAY,
    'USE_SWA': USE_SWA, 'SWA_START_EPOCH': SWA_START_EPOCH,
    'AUG_WEIGHT': AUG_WEIGHT, 'MAX_GRAD_NORM': MAX_GRAD_NORM,
}

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"\nResuming from checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if USE_EMA and 'ema' in ckpt and ckpt['ema'] is not None:
        ema.shadow = {k: v.to(device) for k, v in ckpt['ema'].items()}
    if USE_SWA and 'swa' in ckpt and ckpt['swa'] is not None:
        swa.shadow = ckpt['swa']['shadow']
        swa.n = ckpt['swa']['n']
    start_epoch        = ckpt['epoch']
    best_top1          = ckpt.get('best_top1', 0.0)
    best_epoch         = ckpt.get('best_epoch', 0)
    best_strategy      = ckpt.get('best_strategy', 'none')
    epoch_log          = ckpt.get('epoch_log', [])
    patience_counter   = ckpt.get('patience_counter', 0)
    print(f"Resumed at epoch {start_epoch}, best Top-1 so far: {best_top1:.4f} ({best_strategy})")

print(f"\nTraining epochs {start_epoch+1} to {NUM_EPOCHS}")
print("=" * 110)
print(f" Ep   TrLoss   CE     Marg    EMA-Top1  EMA-T3  EMA-T5  EMA-MacF1  EMA-WtF1  | SWA-Top1     LR")
print("=" * 110)

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = total_ce = total_marg = 0.0
    nb = 0
    epoch_start = time.time()
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for step, batch in enumerate(pbar):
        ids   = batch['input_ids'].to(device, non_blocking=True)
        masks = batch['attention_mask'].to(device, non_blocking=True)
        lbls  = batch['label'].to(device, non_blocking=True)
        sw    = batch['weight'].to(device, non_blocking=True)
        B, K, L = ids.shape
        flat_ids   = ids.view(B * K, L)
        flat_masks = masks.view(B * K, L)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            scores = model(input_ids=flat_ids, attention_mask=flat_masks).logits.squeeze(-1).view(B, K).float()
            loss, ce_loss, margin_loss = weighted_listwise_loss(
                scores, lbls, sw, class_weights,
                LABEL_SMOOTHING, USE_HARD_NEG, MARGIN, LAMBDA_MARGIN
            )

        (loss / GRAD_ACCUM).backward()

        if USE_FGM:
            fgm.attack()
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                scores_adv = model(input_ids=flat_ids, attention_mask=flat_masks).logits.squeeze(-1).view(B, K).float()
                loss_adv, _, _ = weighted_listwise_loss(
                    scores_adv, lbls, sw, class_weights,
                    LABEL_SMOOTHING, USE_HARD_NEG, MARGIN, LAMBDA_MARGIN
                )
            (loss_adv / GRAD_ACCUM).backward()
            fgm.restore()

        total_loss += loss.item()
        total_ce   += ce_loss.item()
        total_marg += margin_loss.item()
        nb += 1

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            if USE_EMA:
                ema.update(model)

        if nb % 50 == 0:
            pbar.set_postfix(loss=f"{total_loss/nb:.3f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    pbar.close()
    epoch_time = time.time() - epoch_start
    print(f"  Epoch {epoch+1} train time: {epoch_time/60:.1f} min")

    avg_loss = total_loss / nb
    avg_ce   = total_ce   / nb
    avg_marg = total_marg / nb

    if USE_EMA:
        ema.apply_shadow(model)
    eval_results = evaluate(model, tokenizer, val_queries, val_labels_list, corpus_texts, MAX_LENGTH)
    if USE_EMA:
        ema.restore(model)

    swa_top1 = None
    if USE_SWA and (epoch + 1) >= SWA_START_EPOCH:
        swa.update(model)
        swa.apply(model)
        swa_eval = evaluate(model, tokenizer, val_queries, val_labels_list, corpus_texts, MAX_LENGTH)
        swa_top1 = swa_eval['top1']
        swa.restore(model)

    cur_lr = optimizer.param_groups[0]['lr']
    log_entry = {
        'epoch': epoch + 1,
        'train_loss': avg_loss, 'ce_loss': avg_ce, 'margin_loss': avg_marg,
        'lr': cur_lr,
        'ema_top1': eval_results['top1'], 'ema_top3': eval_results['top3'], 'ema_top5': eval_results['top5'],
        'ema_macro_f1': eval_results['macro_f1'], 'ema_weighted_f1': eval_results['weighted_f1'],
        'swa_top1': swa_top1,
    }
    epoch_log.append(log_entry)

    swa_str = f"{swa_top1:.4f}" if swa_top1 is not None else "  -   "
    print(f"  {epoch+1:>2}  {avg_loss:6.4f}  {avg_ce:5.3f}  {avg_marg:5.3f}   "
          f"{eval_results['top1']:.4f}    {eval_results['top3']:.4f}  {eval_results['top5']:.4f}  "
          f"{eval_results['macro_f1']:.4f}     {eval_results['weighted_f1']:.4f}     {swa_str}     {cur_lr:.2e}")

    candidates = [('ema', eval_results['top1'])]
    if swa_top1 is not None:
        candidates.append(('swa', swa_top1))
    cand_strategy, cand_top1 = max(candidates, key=lambda x: x[1])

    is_best = cand_top1 > best_top1
    if is_best:
        best_top1     = cand_top1
        best_epoch    = epoch + 1
        best_strategy = cand_strategy
        patience_counter = 0

        if cand_strategy == 'ema' and USE_EMA:
            ema.apply_shadow(model)
            torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, BEST_MODEL_PATH)
            ema.restore(model)
        elif cand_strategy == 'swa' and USE_SWA:
            swa.apply(model)
            torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, BEST_MODEL_PATH)
            swa.restore(model)
        else:
            torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, BEST_MODEL_PATH)
        print(f"  *** New best Top-1 = {best_top1:.4f}  ({best_strategy})  ***")
    else:
        patience_counter += 1

    torch.save({
        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'ema': {k: v.cpu() for k, v in ema.shadow.items()} if USE_EMA else None,
        'swa': ({'shadow': swa.shadow, 'n': swa.n} if USE_SWA and swa.has_data() else None),
        'best_top1': best_top1,
        'best_epoch': best_epoch,
        'best_strategy': best_strategy,
        'epoch_log': epoch_log,
        'patience_counter': patience_counter,
    }, CHECKPOINT_PATH)

    pd.DataFrame(epoch_log).to_csv(os.path.join(RESULTS_DIR, "epoch_log.csv"), index=False)

    if os.path.exists(BEST_MODEL_PATH):
        shutil.copy(BEST_MODEL_PATH, os.path.join(RESULTS_DIR, "best_model.pt"))
    partial_results = {
        'seed': SEED,
        'best_top1': best_top1,
        'best_epoch': best_epoch,
        'best_strategy': best_strategy,
        'completed_epochs': epoch + 1,
        'total_epochs_planned': NUM_EPOCHS,
        'config': config_dict,
        'epoch_log': epoch_log,
    }
    with open(os.path.join(RESULTS_DIR, "results.json"), 'w') as f:
        json.dump(partial_results, f, indent=2, default=str)
    print(f"  [export refreshed] best_model.pt + results.json updated in {RESULTS_DIR}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\n{'=' * 110}")
print(f"Best Top-1: {best_top1:.4f} at epoch {best_epoch}  via  {best_strategy}")
print(f"Best model saved to: {BEST_MODEL_PATH}")

Class weights (sqrt inv-effective-freq, normalized to mean=1):
     11  raw=  581  eff=  523.0  weight=0.811  Agriculture, Forestry, Fishing and Hunting
     21  raw=  361  eff=  335.0  weight=1.014  Mining, Quarrying, and Oil and Gas Extraction
     22  raw=  137  eff=  128.5  weight=1.636  Utilities
     23  raw=  504  eff=  468.5  weight=0.857  Construction
  31-33  raw= 5190  eff= 4861.0  weight=0.266  Manufacturing
     42  raw= 1143  eff= 1062.0  weight=0.569  Wholesale Trade
  44-45  raw= 1276  eff= 1179.5  weight=0.540  Retail Trade
  48-49  raw=  709  eff=  647.0  weight=0.729  Transportation and Warehousing
     51  raw=  530  eff=  485.5  weight=0.842  Information
     52  raw=  588  eff=  545.0  weight=0.795  Finance and Insurance
     53  raw=  332  eff=  306.0  weight=1.060  Real Estate and Rental and Leasing
     54  raw=  700  eff=  654.5  weight=0.725  Professional, Scientific, and Technical Servic
     55  raw=   42  eff=   39.5  weight=2.952  Management of Companies 

Epoch 1/8:   0%|          | 0/3733 [00:00<?, ?it/s]

  Epoch 1 train time: 84.6 min
   1  1.7666  1.623  0.718   0.5134    0.6770  0.7750  0.5038     0.5271       -        1.06e-06
  *** New best Top-1 = 0.5134  (ema)  ***
  [export refreshed] best_model.pt + results.json updated in results_naics2_large_run3_seed42


Epoch 2/8:   0%|          | 0/3733 [00:00<?, ?it/s]

  Epoch 2 train time: 84.7 min
   2  1.2897  1.158  0.657   0.6812    0.8632  0.9217  0.6088     0.6706       -        9.72e-07
  *** New best Top-1 = 0.6812  (ema)  ***
  [export refreshed] best_model.pt + results.json updated in results_naics2_large_run3_seed42


...

  Epoch 6 train time: 84.8 min
   6  1.0206  0.929  0.457   0.6918    0.8836  0.9189  0.6195     0.6856     0.6996     1.77e-07
  *** New best Top-1 = 0.6996  (swa)  ***
  [export refreshed] best_model.pt + results.json updated in results_naics2_large_run3_seed42


Epoch 7/8:   0%|          | 0/3733 [00:00<?, ?it/s]

  Epoch 7 train time: 84.7 min
   7  0.9925  0.906  0.434   0.6911    0.8759  0.9203  0.6203     0.6860     0.6982     4.63e-08
  [export refreshed] best_model.pt + results.json updated in results_naics2_large_run3_seed42


Epoch 8/8:   0%|          | 0/3733 [00:00<?, ?it/s]

  Epoch 8 train time: 84.7 min
   8  0.9749  0.891  0.419   0.6925    0.8766  0.9210  0.6250     0.6886     0.6982     0.00e+00
  [export refreshed] best_model.pt + results.json updated in results_naics2_large_run3_seed42

Best Top-1: 0.6996 at epoch 6  via  swa
Best model saved to: /content/drive/MyDrive/naics2_large_run3_checkpoints_seed42/best_model.pt


In [12]:
print("Reloading best (EMA-or-SWA) weights for final evaluation...")
state = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(state)

final = evaluate(model, tokenizer, val_queries, val_labels_list, corpus_texts, MAX_LENGTH)
print(f"\n=== FINAL VALIDATION (seed={SEED}, strategy={best_strategy}) ===")
print(f"  Top-1: {final['top1']:.4f}")
print(f"  Top-3: {final['top3']:.4f}")
print(f"  Top-5: {final['top5']:.4f}")
print(f"  Macro F1: {final['macro_f1']:.4f}")
print(f"  Weighted F1: {final['weighted_f1']:.4f}")

results = {
    'seed': SEED,
    'best_top1': best_top1,
    'best_epoch': best_epoch,
    'best_strategy': best_strategy,
    'final': {k: v for k, v in final.items() if k != 'all_preds'},
    'config': config_dict,
    'epoch_log': epoch_log,
}
with open(os.path.join(RESULTS_DIR, "results.json"), 'w') as f:
    json.dump(results, f, indent=2, default=str)

shutil.copy(BEST_MODEL_PATH, os.path.join(RESULTS_DIR, "best_model.pt"))
zip_path = shutil.make_archive(f"naics2_large_run3_seed{SEED}", 'zip', RESULTS_DIR)
print(f"\nResults archived: {zip_path}")
print(f"Contents of {RESULTS_DIR}:")
for f in os.listdir(RESULTS_DIR):
    size_mb = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
    print(f"\nDownload triggered for: {zip_path}")
except Exception as e:
    print(f"\nNot downloading - results at: {os.path.abspath(zip_path)}  ({e})")

Reloading best (EMA-or-SWA) weights for final evaluation...

=== FINAL VALIDATION (seed=42, strategy=swa) ===
  Top-1: 0.6996
  Top-3: 0.8850
  Top-5: 0.9267
  Macro F1: 0.6270
  Weighted F1: 0.6918

Results archived: /content/naics2_large_run3_seed42.zip
Contents of results_naics2_large_run3_seed42:
  results.json  (0.0 MB)
  epoch_log.csv  (0.0 MB)
  best_model.pt  (1740.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download triggered for: /content/naics2_large_run3_seed42.zip


In [13]:
state = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(state)
final2 = evaluate(model, tokenizer, val_queries, val_labels_list, corpus_texts, MAX_LENGTH)
print(f"\nRe-eval of best_model.pt:")
print(f"  Top-1: {final2['top1']:.4f}")
print(f"  Top-3: {final2['top3']:.4f}")
print(f"  Top-5: {final2['top5']:.4f}")
print(f"  Macro F1: {final2['macro_f1']:.4f}")
print(f"  Weighted F1: {final2['weighted_f1']:.4f}")


Re-eval of best_model.pt:
  Top-1: 0.6996
  Top-3: 0.8850
  Top-5: 0.9267
  Macro F1: 0.6270
  Weighted F1: 0.6918


In [14]:
if os.path.exists(BEST_MODEL_PATH):
    shutil.copy(BEST_MODEL_PATH, os.path.join(RESULTS_DIR, "best_model.pt"))
    print(f"Refreshed best_model.pt -> {RESULTS_DIR}")
else:
    print(f"WARNING: no best_model.pt at {BEST_MODEL_PATH}")

print(f"\nContents of {RESULTS_DIR}:")
for fn in os.listdir(RESULTS_DIR):
    size_mb = os.path.getsize(os.path.join(RESULTS_DIR, fn)) / 1e6
    print(f"  {fn}  ({size_mb:.1f} MB)")

zip_path = shutil.make_archive(f"naics2_large_run3_seed{SEED}_latest", 'zip', RESULTS_DIR)
print(f"\nLatest export: {zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
    print(f"Download triggered.")
except Exception as e:
    print(f"Not in Colab - file at: {os.path.abspath(zip_path)}  ({e})")


Refreshed best_model.pt -> results_naics2_large_run3_seed42

Contents of results_naics2_large_run3_seed42:
  results.json  (0.0 MB)
  epoch_log.csv  (0.0 MB)
  best_model.pt  (1740.4 MB)

Latest export: /content/naics2_large_run3_seed42_latest.zip  (1501.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.
